# Lesson 16: Project workshop — build your own AI app 🚀
**Goal:** use what you've learned in Lessons 1–15 to build a small AI app **with a web interface**.

**⏱️ Timing:** choose a project 10' | build 35' | test & improve 15'

**📋 Project requirements**
1. A working AI app (with an interface)
2. A **prompt log**: which prompts you tried and why you changed them
3. A clear label: which parts the AI did and which parts you did
4. At least one **mistake or limitation** of the AI that you found

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## Pick a template (or invent your own!)
| Template | Uses lessons | Difficulty |
|---|---|---|
| **A. Study buddy:** a tutor for one subject that gives hints, not answers | 12, 14 | ⭐ |
| **B. Picture-book maker:** topic in → story + illustration out | 10, 12, 13 | ⭐⭐ |
| **C. Rhymes + watermark:** your AI writes rhymes and can detect whether it wrote a text | 5, 15 | ⭐⭐ |
| **D. Your own idea:** school news generator, flashcard maker, role-play a historical figure… | all | ⭐⭐⭐ |

## Step 1: Install the Gradio interface tool and load the model
Gradio lets you build a web interface in a few lines of code.

In [ ]:
!pip -q install gradio
import gradio as gr

# ⏳ The first run downloads the model (about 1–3 minutes). Be patient!
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if torch.cuda.is_available() else "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")

def ask(prompt, system="You are a friendly assistant. Answer in simple English.",
        max_new_tokens=300, temperature=0.7):
    """Send a prompt to the AI and return its answer."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens,
                       do_sample=temperature > 0, temperature=temperature if temperature > 0 else None)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("✅ Model loaded:", MODEL)

## Step 2: The simplest Gradio example
Get this "compliment generator" running first to see how Gradio works.

In [ ]:
def praise(name, thing):
    return ask(f"Give an over-the-top, cute compliment to a classmate called {name} because they {thing}. Max 3 sentences.")

gr.Interface(fn=praise,
             inputs=[gr.Textbox(label="Name"), gr.Textbox(label="What good thing did they do?")],
             outputs=gr.Textbox(label="The AI's compliment"),
             title="🌈 Compliment Generator").launch()

## Template A: Study buddy 📚
Change `SUBJECT` and `RULES` to build your own tutor.

In [ ]:
SUBJECT = "Year 9 science"
RULES = f"""You are a patient {SUBJECT} tutor. Your student is about 13 years old.
Rules:
1. Never give the final answer directly
2. Give only one small hint at a time, or ask a guiding question
3. When the student gets something right, praise them specifically
4. If the question isn't about {SUBJECT}, politely steer back to the topic"""

def tutor(message, history):
    return ask(message, system=RULES)

gr.ChatInterface(tutor, title=f"🧑‍🏫 {SUBJECT} study buddy").launch()

## Template B: Picture-book maker 🎨 (needs a GPU)

In [ ]:
!pip -q install diffusers
from diffusers import AutoPipelineForText2Image
painter = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")

def make_page(topic, style):
    story = ask(f"Write a children's story of at most 80 words about: {topic}")
    scene = ask(f"Describe the most exciting picture in this story in one sentence. Output only that sentence: {story}",
                max_new_tokens=60)
    image = painter(scene + f", {style}", num_inference_steps=1, guidance_scale=0.0).images[0]
    return story, image

gr.Interface(fn=make_page,
             inputs=[gr.Textbox(label="Story topic"),
                     gr.Dropdown(["watercolor", "pixel art", "3D cartoon", "crayon drawing"], label="Art style")],
             outputs=[gr.Textbox(label="Story"), gr.Image(label="Illustration")],
             title="📖 Picture-Book Maker").launch()

## Template C: The rhyme writer that signs its work ✒️
Copy the watermark code from Lesson 15 and build an app with two tabs: one writes rhymes, one detects them.

In [ ]:
# 👉 Paste the Lesson 15 code here: corpus, words, join, next_word, SECRET, is_green, generate, detect

# Then:
# with gr.Blocks() as app:
#     with gr.Tab("Write"):
#         start = gr.Textbox(label="First word"); out = gr.Textbox()
#         gr.Button("Write!").click(lambda s: join(generate(s.lower(), 40, True)), start, out)
#     with gr.Tab("Detect"):
#         txt = gr.Textbox(label="Paste text"); res = gr.Textbox()
#         gr.Button("Detect").click(lambda t: detect(re.findall(r"[a-z']+|[.,?]", t.lower())), txt, res)
# app.launch()

## 📝 Project log (required!)

In [ ]:
project = {
    "Team members": "",
    "App name": "",
    "Problem it solves / who it's for": "",
    "Prompt version 1": "",
    "Prompt version 2 (what changed and why)": "",
    "Final prompt": "",
    "Parts the AI did": "",
    "Parts we did ourselves": "",
    "AI mistakes or limitations we found": "",
    "What we'd improve next": "",
}
for k, v in project.items(): print(f"[{k}] {v}")

## ✅ Testing checklist
Before you hand in, ask **another group** to test your app:
- [ ] Do normal inputs give good results?
- [ ] Do weird inputs (blank, gibberish, very long) break it?
- [ ] Can you "trick" it into doing something it shouldn't (e.g. making the tutor give the answer)?
- [ ] Is there anything factually wrong or biased in the output?